# Análisis de la operación logística

Notebook complementario de la app de consola. Reutiliza **los mismos repositorios y servicios** (`src/`), así los números coinciden con el menú *Estadísticas*.

> Si `data/` está vacío, ejecutá la celda opcional de datos de ejemplo o usá la opción **5** del menú principal.

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete src/ cuando el notebook se abre desde la raíz del proyecto
RAIZ = Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
import pandas as pd

from src.repositories import PaqueteRepository, RepartidorRepository, RepartoRepository
from src.services import AsignacionService, ClimaService, EstadisticasService, SeederService

paquetes, repartidores, repartos = PaqueteRepository(), RepartidorRepository(), RepartoRepository()
clima = ClimaService()
asignacion = AsignacionService(paquetes, repartidores, repartos, clima)
stats = EstadisticasService(paquetes, repartidores, repartos, asignacion)

### (Opcional) Generar datos de ejemplo
Borra los CSV actuales y crea un escenario reproducible.

In [ ]:
if not paquetes.get_all():
    print(SeederService(paquetes, repartidores, repartos, asignacion).cargar())
else:
    print("Ya hay datos cargados; no se regeneran.")

## 1. Datos base

In [ ]:
df_paquetes = stats.df_paquetes()
df_repartidores = stats.df_repartidores()
df_repartos = stats.df_repartos()
print(f"Paquetes: {len(df_paquetes)} | Repartidores: {len(df_repartidores)} | Repartos: {len(df_repartos)}")
df_repartos.head()

In [ ]:
stats.resumen()

## 2. KPI 1 — Tasa de ocupación de repartidores

- Por repartidor: $\text{ocupación} = \max\left(\frac{\text{carga kg}}{\text{cap. kg}}, \frac{\text{carga dm}^3}{\text{cap. dm}^3}\right) \times 100$ (la dimensión que limita).
- Global ponderada: $\frac{\sum \text{carga kg}}{\sum \text{cap. kg}} \times 100$ — un vehículo grande pesa más que una moto.
- Solo cuentan los repartos **activos** (pendientes y en camino).

In [ ]:
ocupacion, global_pct = stats.kpi_ocupacion()
print(f"Ocupación global ponderada: {global_pct} %")
ocupacion

In [ ]:
# Ocupación promedio por tipo de vehículo
ocupacion.groupby("vehiculo")["ocupacion_%"].agg(["count", "mean", "max"]).round(1)

## 3. KPI 2 — Costo medio por zona

$\text{costo promedio}(z) = \frac{\sum \text{costo total}}{\text{cantidad de repartos de } z}$ (excluye cancelados).

`costo total = (tarifa base de la zona + $150/kg + $8/dm³) × (1 + recargo clima %)`

In [ ]:
stats.kpi_costo_por_zona()

## 4. KPI 3 — Distribución de estados

In [ ]:
stats.kpi_estados()

## 5. Impacto del clima en el costo

In [ ]:
no_cancelados = df_repartos[df_repartos["estado"] != "Cancelado"]
impacto = no_cancelados.assign(recargo_pesos=no_cancelados["costo_total"] - no_cancelados["costo_base"])
impacto.groupby(["zona", "clima"])[["costo_base", "recargo_pesos", "costo_total"]].sum().round(2)

## 6. Gráficos

In [ ]:
ruta = stats.generar_graficos()
print(f"Guardado en {ruta}")
plt.show()